## Lecture 4 Exercises: PRS

1) Discovery summary-stat QC (15 pts)
a. Column/format validation
Verify required fields exist: SNP, A1, A2, BETA, SE, P, N, CHR,BP
b. Missingness check:
Report number of rows with any missing SNP/A1/A2/BETA/SE/P/N; remove them.
c. Range checks:
• Confirm 0 ≤ P ≤ 1
• Confirm SE > 0
• flag extreme |BETA| outliers
d. Duplicate SNP IDs:
Count duplicates and drop any duplicated values (keep the smallest P).

Name your QC'd version: discovery_gwas_sim_sumstats_cleaned.tsv
Include the discovery_gwas_sim_sumstats_cleaned.tsv in your submission.

In [21]:
import pandas as pd
import numpy as np

# Load the data 
print("Loading GWAS summary stats...")
df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats.tsv.gz', sep='\t')
initial_rows = len(df)
print(f"Initial row count: {initial_rows}\n")

# ---------------------------------------------------------
# 1a. Column/format validation
# ---------------------------------------------------------
required_cols = ['SNP', 'A1', 'A2', 'BETA', 'SE', 'P', 'N', 'CHR', 'BP']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"1a. Missing columns: {missing_cols}")
else:
    print("1a. All required columns are present.\n")



Loading GWAS summary stats...
Initial row count: 1462262

1a. All required columns are present.



In [22]:
# ---------------------------------------------------------
# 1b. Missingness check
# ---------------------------------------------------------
# Drop rows with missing values in the critical columns
df_clean = df.dropna(subset=['SNP', 'A1', 'A2', 'BETA', 'SE', 'P', 'N'])
missing_dropped = initial_rows - len(df_clean)

print(f"1b. Missingness check:")
print(f"Rows removed due to missing data: {missing_dropped}")
print(f"Rows remaining: {len(df_clean)}\n")

1b. Missingness check:
Rows removed due to missing data: 1989
Rows remaining: 1460273



In [23]:
# ---------------------------------------------------------
# 1c. Range checks
# ---------------------------------------------------------
# Confirm 0 <= P <= 1 and SE > 0
valid_p = (df_clean['P'] >= 0) & (df_clean['P'] <= 1)
valid_se = df_clean['SE'] > 0

# Apply the filters
pre_range_count = len(df_clean)
df_clean = df_clean[valid_p & valid_se]
range_dropped = pre_range_count - len(df_clean)

print(f"1c. Range checks:")
print(f"Rows removed (invalid P or SE): {range_dropped}")

# Flag extreme |BETA| outliers 
# (Using > 3 Standard Deviations as the definition of "extreme")
beta_mean = df_clean['BETA'].mean()
beta_std = df_clean['BETA'].std()
extreme_beta_mask = np.abs(df_clean['BETA'] - beta_mean) > (3 * beta_std)
extreme_betas = df_clean[extreme_beta_mask]

print(f"Flagged {len(extreme_betas)} extreme |BETA| outliers (> 3 SDs).")
print()



1c. Range checks:
Rows removed (invalid P or SE): 1240
Flagged 28471 extreme |BETA| outliers (> 3 SDs).



In [24]:
# ---------------------------------------------------------
# 1d. Duplicate SNP IDs
# ---------------------------------------------------------
# Count all rows that share a SNP ID with another row
duplicate_count = df_clean.duplicated(subset=['SNP'], keep=False).sum()
print(f"1d. Duplicate SNP IDs:")
print(f"Found {duplicate_count} duplicated SNP rows.")

# Sort by P-value (ascending) so the smallest P is at the top
df_clean = df_clean.sort_values('P')

# Drop duplicates, keeping the first (which has the smallest P due to sorting)
pre_dedup_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['SNP'], keep='first')
dedup_dropped = pre_dedup_count - len(df_clean)

print(f"Dropped {dedup_dropped} duplicate records (kept smallest P).")
print(f"Final cleaned row count: {len(df_clean)}\n")

# ---------------------------------------------------------
# Save Output
# ---------------------------------------------------------
output_file = '/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats_cleaned.tsv'
df_clean.to_csv(output_file, sep='\t', index=False)


1d. Duplicate SNP IDs:
Found 8716 duplicated SNP rows.
Dropped 4358 duplicate records (kept smallest P).
Final cleaned row count: 1454675



2) Harmonization with target BIM (15 pts)
a. SNP overlap with target
    Intersect SNP list in discovery_gwas_sim_sumstats_cleaned.tsv with target .bim SNP IDs; report overlap count and % retained.
b. Allele matching
    For overlapping SNPs, verify base A1/A2 match target BIM alleles (order may differ).
c. Allele flipping logic
    If base A1/A2 are swapped relative to BIM, flip effect direction: BETA := -BETA and swap alleles.
    Report how many SNPs were flipped.
d. Ambiguous strand SNP handling
Identify A/T and C/G SNPs and remove them.

Output a file named prs_weights_harmonized.tsv after your QC and harmonization steps with these columns:
SNP — SNP ID (matches the target .bim SNP ID)
A1 — effect allele (aligned to the target allele coding)
Note: This is not the A1 in the "ref allele sense" it is either a risk allele if beta>0 or a
protective allele if beta<0
BETA — SNP weight (with sign flipped if alleles were swapped)
P — GWAS p-value (handy for thresholding, optional for scoring)

Include the prs_weights_harmonized_qc.tsv in your submission.


In [33]:
import pandas as pd

# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------
print("Loading datasets for harmonization...")
# 1. Load the cleaned base stats from Step 1
base_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats_cleaned.tsv', sep='\t')
initial_base_count = len(base_df)

# 2. Load the target BIM file. 
# BIM files have no header and are space/tab separated.
# The standard PLINK 6 columns are: CHR, SNP, CM, BP, A1 (alt), A2 (ref)
bim_cols = ['BIM_CHR', 'SNP', 'BIM_CM', 'BIM_BP', 'BIM_A1', 'BIM_A2']
bim_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final.bim', sep='\s+', header=None, names=bim_cols)

# ---------------------------------------------------------
# 2a. SNP overlap with target
# ---------------------------------------------------------
# Intersect the base summary stats with the target BIM on the 'SNP' column
merged_df = pd.merge(base_df, bim_df[['SNP', 'BIM_A1', 'BIM_A2']], on='SNP', how='inner')

overlap_count = len(merged_df)
percent_retained = (overlap_count / initial_base_count) * 100

print(f"2a. SNP Overlap:")
print(f"    - Original base SNPs: {initial_base_count}")
print(f"    - Overlapping SNPs: {overlap_count}")
print(f"    - % Retained: {percent_retained:.2f}%\n")



<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
/var/folders/_7/04cflckd1md8652444n3682j8w0jjt/T/ipykernel_75468/409196043.py:15: SyntaxWarning: invalid escape sequence '\s'
  bim_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final.bim', sep='\s+', header=None, names=bim_cols)


Loading datasets for harmonization...
2a. SNP Overlap:
    - Original base SNPs: 1454675
    - Overlapping SNPs: 1068130
    - % Retained: 73.43%



In [34]:
# ---------------------------------------------------------
# 2b & 2c. Allele matching and flipping logic
# ---------------------------------------------------------
# Condition 1: Exact Match
exact_match = (merged_df['A1'] == merged_df['BIM_A1']) & (merged_df['A2'] == merged_df['BIM_A2'])

# Condition 2: Swapped 
flip_match = (merged_df['A1'] == merged_df['BIM_A2']) & (merged_df['A2'] == merged_df['BIM_A1'])
flipped_count = flip_match.sum()

# Apply the flipping logic:
merged_df.loc[flip_match, 'BETA'] = -merged_df.loc[flip_match, 'BETA']
merged_df.loc[flip_match, 'A1'] = merged_df.loc[flip_match, 'BIM_A1']
merged_df.loc[flip_match, 'A2'] = merged_df.loc[flip_match, 'BIM_A2']

# Discard anything that isn't an exact match or a perfect flip
valid_snps = exact_match | flip_match
invalid_count = len(merged_df) - valid_snps.sum()
merged_df = merged_df[valid_snps].copy()

print(f"2b & 2c. Allele Matching and Flipping:")
print(f"    - Exact allele matches: {exact_match.sum()}")
print(f"    - Alleles swapped & BETAs flipped: {flipped_count}")
print(f"    - Incompatible alleles removed: {invalid_count}\n")

2b & 2c. Allele Matching and Flipping:
    - Exact allele matches: 1043501
    - Alleles swapped & BETAs flipped: 19229
    - Incompatible alleles removed: 5400



In [35]:
# ---------------------------------------------------------
# 2d. Ambiguous strand SNP handling 
# ---------------------------------------------------------
# Identify A/T and C/G pairs
def is_ambiguous(a1, a2):
    alleles = set([a1, a2])
    return alleles == {'A', 'T'} or alleles == {'C', 'G'}

ambiguous_mask = merged_df.apply(lambda row: is_ambiguous(row['A1'], row['A2']), axis=1)
ambig_count = ambiguous_mask.sum()

# Remove them
merged_df = merged_df[~ambiguous_mask].copy()

print(f"2d. Ambiguous Strand Handling:")
print(f"    - Removed {ambig_count} ambiguous (A/T or C/G) SNPs.\n")
print(f"Final harmonized SNP count: {len(merged_df)}\n")

2d. Ambiguous Strand Handling:
    - Removed 82059 ambiguous (A/T or C/G) SNPs.

Final harmonized SNP count: 980671



In [ ]:
# ---------------------------------------------------------
# Output Formatting
# ---------------------------------------------------------

final_df = merged_df[['SNP', 'A1', 'BETA', 'P']]

output_file = '/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized.tsv'
final_df.to_csv(output_file, sep='\t', index=False)
print(f"Successfully saved harmonized weights to: {output_file}")

Successfully saved harmonized weights to: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized.tsv


In [51]:
df_clean

,SNP,CHR,BP,A1,A2,BETA,SE,P,N
484489,rs7726558,5,118940497,C,G,8.647514e-02,0.010003,5.389593e-18,20000
1063088,rs1171079,13,35383796,T,C,8.143263e-02,0.010050,5.363173e-16,20000
1364598,rs6137023,20,20329302,A,G,-8.228487e-02,0.010164,5.684493e-16,20000
457640,rs6893752,5,60410669,A,G,8.211613e-02,0.010362,2.280970e-15,20000
1046683,rs7300238,12,124322935,T,C,7.840982e-02,0.010227,1.765128e-14,20000
...,...,...,...,...,...,...,...,...,...
297997,rs10934239,3,114851173,T,C,5.400439e-08,0.010336,9.999958e-01,20000
87775,rs10801173,1,191321108,T,C,-4.289612e-08,0.012261,9.999972e-01,20000
852779,rs4749499,10,30201393,C,A,-2.801317e-08,0.016331,9.999986e-01,20000
982190,rs11061851,12,1541441,G,A,-7.154353e-09,0.010000,9.999994e-01,20000


3) PRS-ready output sanity checks (10 pts)
a. Final weight file integrity
Confirm final file prs_weights_harmonized.tsv has unique SNP IDs and no missing BETA.
b. Spot-check allele alignment
Randomly sample ~10 SNPs and manually verify (SNP ID, alleles, flip status) against target BIM.

Include the prs_weights_harmonized_qcd.tsv in your submission.




In [ ]:

# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------
harmonized_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized.tsv', sep='\t')

# ---------------------------------------------------------
# 3a. Final weight file integrity
# ---------------------------------------------------------
# 1. Confirm unique SNP IDs
duplicate_count = harmonized_df.duplicated(subset=['SNP']).sum()
is_unique = (duplicate_count == 0)

# 2. Confirm no missing BETAs
missing_beta_count = harmonized_df['BETA'].isna().sum()
no_missing_betas = (missing_beta_count == 0)

print("\n--- 3a. Final Weight File Integrity ---")
print(f"Unique SNP IDs confirmed: {is_unique} (Duplicates found: {duplicate_count})")
print(f"No missing BETAs confirmed: {no_missing_betas} (Missing found: {missing_beta_count})")



--- 3a. Final Weight File Integrity ---
Unique SNP IDs confirmed: True (Duplicates found: 0)
No missing BETAs confirmed: True (Missing found: 0)


In [42]:
# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------

base_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats_cleaned.tsv', sep='\t')

bim_cols = ['BIM_CHR', 'SNP', 'BIM_CM', 'BIM_BP', 'BIM_A1', 'BIM_A2']
bim_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final.bim', sep='\s+', header=None, names=bim_cols)

# ---------------------------------------------------------
# 3b. Spot-check allele alignment
# ---------------------------------------------------------
print("\n--- 3b. Spot-Check Allele Alignment (n=10) ---")

# Randomly sample 10 SNPs from the harmonized data
sample_snps = harmonized_df.sample(n=10, random_state=42)

# Merge the sample back with the Base and BIM data to create a comparison table
spot_check_df = sample_snps[['SNP', 'A1', 'BETA']].rename(columns={'A1': 'Harmonized_A1', 'BETA': 'Harmonized_BETA'})
spot_check_df = spot_check_df.merge(base_df[['SNP', 'A1', 'A2', 'BETA']].rename(columns={'A1': 'Base_A1', 'A2': 'Base_A2', 'BETA': 'Base_BETA'}), on='SNP')
spot_check_df = spot_check_df.merge(bim_df[['SNP', 'BIM_A1', 'BIM_A2']], on='SNP')

# Reorder columns for easy side-by-side reading
spot_check_df = spot_check_df[['SNP', 'Base_A1', 'Base_A2', 'Base_BETA', 'BIM_A1', 'BIM_A2', 'Harmonized_A1', 'Harmonized_BETA']]

# Print the table for manual verification
print(spot_check_df.to_string(index=False))

# ---------------------------------------------------------
# Output Formatting
# ---------------------------------------------------------
# Save the exact same DataFrame to the new required filename
output_file = '/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized_qcd.tsv'
harmonized_df.to_csv(output_file, sep='\t', index=False)
print(f"\nSuccessfully saved final verified file to: {output_file}")

<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
/var/folders/_7/04cflckd1md8652444n3682j8w0jjt/T/ipykernel_75468/702392697.py:8: SyntaxWarning: invalid escape sequence '\s'
  bim_df = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final.bim', sep='\s+', header=None, names=bim_cols)



--- 3b. Spot-Check Allele Alignment (n=10) ---
       SNP Base_A1 Base_A2  Base_BETA BIM_A1 BIM_A2 Harmonized_A1  Harmonized_BETA
rs10860936       C       T  -0.009002      C      T             C        -0.009002
  rs264748       A       G   0.017374      A      G             A         0.017374
 rs3816792       T       C   0.004321      T      C             T         0.004321
 rs2699006       T       G  -0.000989      T      G             T        -0.000989
  rs738223       A       G  -0.016826      G      A             G         0.016826
rs13436568       T       C   0.004260      T      C             T         0.004260
 rs6743733       C       T   0.009839      C      T             C         0.009839
 rs4480847       G       A  -0.009968      G      A             G        -0.009968
 rs1517944       T       C   0.003626      T      C             T         0.003626
 rs4896326       T       C   0.015871      T      C             T         0.015871

Successfully saved final verified file

4) PRS for a Binary Trait (C+T) + Logistic Regression with Ancestry Covariates (80 pts)
Compute PRS in the target cohort using clumping + P-value thresholding (C+T) from cleaned discovery GWAS summary statistics, then test association with a binary phenotype using logistic regression while adjusting for ancestry covariates (PC/MDS).

Required Inputs:
1. Target genotypes:
HapMap_3_r3_1.bed/.bim/.fam files
2. Cleaned base GWAS summary stats:
discovery_gwas_sim_sumstats_cleaned.tsv
3. Binary phenotype file (case/control):
pheno.tsv with columns: FID IID PHENO
where PHENO is 1 = control, 2 = case (PLINK convention)
4. Ancestry covariates:
covar.tsv with columns FID IID PC1 PC2 ... (from GWAS tutorial)

4.1) LD Clumping (Target LD) (20 pts)
To obtain approximately independent SNPs for polygenic risk score (PRS) construction, perform LD clumping using the target genotype dataset.

a. Using PLINK index SNPs that are in strong LD with an LD threshold of r. = 0.1 within a 250 kb window.
Required inputs:
1. HapMap_3_r3_1 genotypes (the command requires you to include the prefix HapMap_3_r3_1 while ensuring all .bed/.bim/.fam files are in the same working directory)
2. discovery_gwas_sim_sumstats_cleaned.tsv

Expected outputs:
1. clump.clumped file

b. Extract the list of clumped SNPs from clump.clumped and save them in a file called clumped.snps

Include the clumped.snps file in your submission.




In [74]:
# Run clumping in PLINK
! /Users/kmlanderos/Downloads/plink_mac_20250819/plink \
    --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final \
    --clump /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats_cleaned.tsv \
    --clump-r2 0.1 \
    --clump-kb 250 \
    --clump-p1 1 \
    --clump-p2 1 \
    --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clump

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clump.log.
Options in effect:
  --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final
  --clump /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats_cleaned.tsv
  --clump-kb 250
  --clump-p1 1
  --clump-p2 1
  --clump-r2 0.1
  --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clump

18432 MB RAM detected; reserving 9216 MB for main workspace.
1073788 variants loaded from .bim file.
76 people (39 males, 37 females) loaded from .fam.
76 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 76 founders an

In [75]:
! head /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clump.clumped

 CHR    F         SNP         BP        P    TOTAL   NSIG    S05    S01   S001  S0001    SP2
   5    1   rs7726558  118940497   5.39e-18       35     33      1      1      0      0 rs1045241(1),rs1045242(1),rs11064(1),rs7703744(1),rs10478424(1),rs6897978(1),rs6898518(1),rs11749784(1),rs10463469(1),rs257973(1),rs25640(1),rs17453681(1),rs3797371(1),rs246968(1),rs92631(1),rs6864259(1),rs1388107(1),rs13188833(1),rs10519599(1),rs10463719(1),rs2896964(1),rs1574641(1),rs988465(1),rs17145587(1),rs994149(1),rs2431376(1),rs329144(1),rs330196(1),rs330195(1),rs330193(1),rs330192(1),rs780429(1),rs1597713(1),rs874886(1),rs1451809(1)
   5    1   rs6893752   60410669   2.28e-15       68     67      0      1      0      0 rs17332108(1),rs4700397(1),rs17332419(1),rs11740632(1),rs34435485(1),rs3117(1),rs4647150(1),rs4235483(1),rs7726671(1),rs4647113(1),rs4647102(1),rs976630(1),rs4647073(1),rs158570(1),rs158935(1),rs158572(1),rs4647028(1),rs158919(1),rs158914(1),rs7722373(1),rs1382914(1),rs3101879(1),rs26

In [76]:
# Extract the Clumped SNPs
! awk 'NR>1 && NF>0 {print $3}' /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clump.clumped > /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clumped.snps
! head /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clumped.snps
! wc -l /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clumped.snps

rs7726558
rs6893752
rs1955617
rs816499
rs17702090
rs6109854
rs12453759
rs17744427
rs9910653
rs6695824
87103 /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clumped.snps


4.2) Threshold SNPs by GWAS P-value (20 pts)
After LD clumping, create a PRS SNP sets using p-value thresholding (the p-value here refers to the raw p-values from GWAS). For each threshold 𝑇, select SNPs from the cleaned GWAS summary statistics with p ≤ T, then intersect that set with the SNPs retained after clumping.

Use the following p-value thresholds:
5e-8, 1e-6, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0

For each threshold 𝑇, identify all SNPs in the cleaned GWAS summary statistics with P ≤ T, then intersect that SNP list with the set of SNPs retained after LD clumping.

Required inputs:
1. discovery_gwas_sim_sumstats_cleaned.tsv
2. clumped.snps

Expected outputs:
1. For each threshold, generate a file containing the SNPs that:
• pass the specified GWAS P-value cutoff, and
• are present in the clumped SNP list
• For example: prs_5e-8.snps, prs_1e-4.snps

Include all prs_{T}.snp files in your submission



In [54]:
import pandas as pd

# ---------------------------------------------------------
# Define our thresholds and their exact string representations
# ---------------------------------------------------------
thresholds = {
    "5e-8": 5e-8, 
    "1e-6": 1e-6, 
    "1e-4": 1e-4, 
    "1e-3": 1e-3, 
    "1e-2": 1e-2, 
    "0.05": 0.05, 
    "0.1": 0.1, 
    "0.5": 0.5, 
    "1.0": 1.0
}

# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------
print("Loading summary stats and clumped SNPs...")

# 1. Load the cleaned summary statistics
sumstats = pd.read_csv('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/discovery_gwas_sim_sumstats_cleaned.tsv', sep='\t')

# 2. Load the list of clumped SNPs into a Python 'set' for lightning-fast matching
with open('/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/clumped.snps', 'r') as f:
    clumped_snps = set(f.read().splitlines())

print(f"Total SNPs in summary stats: {len(sumstats)}")
print(f"Total SNPs passing LD clumping: {len(clumped_snps)}\n")

# ---------------------------------------------------------
# Mathematical Intersection (Clumping)
# ---------------------------------------------------------
# Pre-filter the dataframe to only include SNPs that survived the clumping step
# This makes the loop much faster!
clumped_df = sumstats[sumstats['SNP'].isin(clumped_snps)]

# ---------------------------------------------------------
# Thresholding Loop
# ---------------------------------------------------------
print("Generating threshold files...")

for t_str, t_val in thresholds.items():
    # Filter the clumped data for P-values less than or equal to the threshold
    valid_snps = clumped_df[clumped_df['P'] <= t_val]['SNP']

    # Save the SNP IDs to a text file 
    out_file = f"/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_{t_str}.snps"
    valid_snps.to_csv(out_file, index=False, header=False)
    
    # Print a status update for your report
    print(f"  - {out_file}: {len(valid_snps)} SNPs retained with P <= {t_str}")


Loading summary stats and clumped SNPs...
Total SNPs in summary stats: 1454675
Total SNPs passing LD clumping: 87103

Generating threshold files...
  - /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_5e-8.snps: 33 SNPs retained with P <= 5e-8
  - /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-6.snps: 60 SNPs retained with P <= 1e-6
  - /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-4.snps: 232 SNPs retained with P <= 1e-4
  - /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-3.snps: 1241 SNPs retained with P <= 1e-3
  - /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-2.snps: 8905 SNPs retained with P <= 1e-2
  - /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.05.snps: 27519 SNPs retained with P <= 0.05
  - /Users

4.3) Compute PRS with PLINK (20 pts)
For each p-value threshold compute PRS score using PLINK
Required inputs:
1. HapMap_3_r3_1 genotypes
(the command requires you to include the prefix HapMap_3_r3_1 while ensuring all
.bed/.bim/.fam files are in the same working directory)
2. pheno.tsv
3. covar.tsv
4. prs_{T}.snps file
5. prs_weights_harmonized_qcd.tsv
Expected outputs:
1. .profile output file per individual

Include all .profile files in your submission



In [68]:
%%bash

# Define the 9 thresholds in a list
for T in "5e-8" "1e-6" "1e-4" "1e-3" "1e-2" "0.05" "0.1" "0.5" "1.0"
do
    echo "Calculating PRS for threshold: $T"
    
    /Users/kmlanderos/Downloads/plink_mac_20250819/plink \
        --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final \
        --score /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized_qcd.tsv 1 2 3 header \
        --extract /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_${T}.snps \
        --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_${T}
done

Calculating PRS for threshold: 5e-8


PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_5e-8.log.
Options in effect:
  --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final
  --extract /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_5e-8.snps
  --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_5e-8
  --score /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized_qcd.tsv 1 2 3 header

18432 MB RAM detected; reserving 9216 MB for main workspace.
1073788 variants loaded from .bim file.
76 people (39 males, 37 females) loaded from .fam.
76 phenotype values loaded from .fam.
--extract: 33 variants remaining.
Using

mismatch, 0 due to allele code mismatch); see
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_5e-8.nopred
for details.


--score: 30 valid predictors loaded.
--score: Results written to
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_5e-8.profile
.
Calculating PRS for threshold: 1e-6
PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-6.log.
Options in effect:
  --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final
  --extract /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-6.snps
  --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-6
  --score /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized_qcd.tsv 1 2 3 header

18432 MB RAM detected

mismatch, 0 due to allele code mismatch); see
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1e-6.nopred
for details.


...

--score: 36296 valid predictors loaded.
--score: Results written to
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.1.profile
.
Calculating PRS for threshold: 0.5
PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.5.log.
Options in effect:
  --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final
  --extract /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.5.snps
  --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.5
  --score /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized_qcd.tsv 1 2 3 header

18432 MB RAM detected; 

mismatch, 0 due to allele code mismatch); see
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.5.nopred
for details.


--score: 66185 valid predictors loaded.
--score: Results written to
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_0.5.profile
.
Calculating PRS for threshold: 1.0
PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1.0.log.
Options in effect:
  --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final
  --extract /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1.0.snps
  --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1.0
  --score /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_weights_harmonized_qcd.tsv 1 2 3 header

18432 MB RAM detected; 

mismatch, 0 due to allele code mismatch); see
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1.0.nopred
for details.


--score: 80551 valid predictors loaded.
--score: Results written to
/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_1.0.profile
.


4.4) Logistic Regression: phenotype ~ covariates + PRS (20 pts)
Next, test whether the PRS is associated with the binary phenotype while adjusting for
ancestry covariates.For each threshold, use the PRS computed in the previous step as an
additional covariate in a logistic regression model.

This requires combining:
• the phenotype file
• the ancestry covariate file
• the PRS values from the corresponding .profile file

You will first need to extract the relevant PRS column from the PLINK profile output and
merge it with the ancestry covariates so that each individual has:
• FID
• IID
• ancestry PCs (or MDS covariates)
• PRS

Then run logistic regression in PLINK using the binary phenotype as the outcome.

Required inputs:
1. HapMap_3_r3_1.bed/.bim/.fam
2. pheno.tsv
3. covar.tsv
4. prs_{T}.profile

Expected outputs:
1. assoc_${T}.assoc.logistic

Include all assoc_${T}.assoc.logistic files in your submission

In [ ]:
# ------------------ Create pheno.tsv ------------------

# 1. Create the file and add the required header
! echo -e "FID\tIID\tPHENO" > /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/pheno.tsv

# 2. Extract columns 1, 2, and 6 from the .fam file and append them
! awk '{print $1 "\t" $2 "\t" $6}' /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final.fam >> /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/pheno.tsv
! head /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/pheno.tsv

FID	IID	PHENO
1328	NA06989	2
1349	NA11843	1
1328	NA06984	2
13291	NA06986	1
1418	NA12272	1
13292	NA07051	2
1421	NA12287	1
1330	NA12340	2
1418	NA12273	1


In [60]:
# Convert TXT into CSV
! awk -v OFS=',' '{$1=$1; print}' /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/covar_mds.txt > /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/covar_mds.csv

In [69]:
# ----------------- Create the Merged Covariate Files -----------------

# 1. Load your base covariates (handles tabs or spaces automatically)
covar_file = '/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/covar_mds.csv' 
covar = pd.read_csv(covar_file, sep=',')

# 2. Define the thresholds
thresholds = ["5e-8", "1e-6", "1e-4", "1e-3", "1e-2", "0.05", "0.1", "0.5", "1.0"]

print("Merging PRS scores with ancestry covariates...")

# 3. Loop through and create a custom covariate file for each threshold
for T in thresholds:
    # Load the specific PRS profile and grab only the IDs and the SCORE
    prs = pd.read_csv(f'/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_{T}.profile', sep='\s+')[['FID', 'IID', 'SCORE']]
    
    # Merge them together, matching perfectly on FID and IID
    merged = pd.merge(covar, prs, on=['FID', 'IID'])
    
    # Save as a temporary text file formatted for PLINK
    out_name = f'/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_{T}.txt'
    merged.to_csv(out_name, sep='\t', index=False)
    
    print(f"  - Successfully created: {out_name}")

print("\nCovariate files are ready for PLINK.")

Merging PRS scores with ancestry covariates...
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_5e-8.txt
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_1e-6.txt
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_1e-4.txt
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_1e-3.txt
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_1e-2.txt
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_0.05.txt
  - Successfully created: /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_

<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
/var/folders/_7/04cflckd1md8652444n3682j8w0jjt/T/ipykernel_75468/1553586467.py:15: SyntaxWarning: invalid escape sequence '\s'
  prs = pd.read_csv(f'/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/prs_{T}.profile', sep='\s+')[['FID', 'IID', 'SCORE']]


In [71]:
! head /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_5e-8.txt

FID	IID	C1	C2	C3	C4	C5	C6	C7	C8	C9	C10	SCORE
1328	NA06989	-0.00892223	-0.00713162	0.00371474	-0.0336405	0.0491909	-0.0292816	0.0353081	-0.0457815	0.0136169	-0.00858229	-0.00176948
1349	NA11843	-0.00177884	-0.0047031	0.0203468	0.00572657	-0.0115828	-0.0251281	-0.00320685	0.040138	-0.0350728	0.0295647	-0.00417765
1328	NA06984	0.00475568	-0.00595558	0.0110259	0.00810543	-0.00415454	-0.0329294	0.0100303	-0.0146246	-0.0011256	0.0146921	-0.000858515
13291	NA06986	0.00193304	-0.000197486	0.0324704	-0.00544899	-0.00537063	0.0267853	0.00591263	-0.0200994	-0.0134265	-0.00394412	-0.0117463
1418	NA12272	-0.00824241	-0.0197354	0.0209867	0.00688452	-0.0277361	0.00614233	-0.0105453	0.0188245	-0.0157143	0.0232067	-0.00656992
13292	NA07051	-0.0028356	0.00767111	-0.0493225	0.0259118	0.0396869	-0.0101806	0.0106866	0.0132911	0.0116252	0.0278922	-0.00287749
1421	NA12287	-0.00449084	-0.0132197	0.0321257	-0.00477584	0.00115071	-0.00577709	-0.000662632	-0.0212702	0.00544242	0.0498349	-0.00175742
1330	NA12340	

In [72]:
%%bash

# ------------------------- Run the Logistic Regression -------------------------

# Define all your absolute paths
WORK_DIR="/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/"
PLINK="/Users/kmlanderos/Downloads/plink_mac_20250819/plink"
BFILE="/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final"
PHENO_FILE="/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/pheno.tsv"

# Loop through all 9 thresholds
for T in "5e-8" "1e-6" "1e-4" "1e-3" "1e-2" "0.05" "0.1" "0.5" "1.0"
do
    echo "========================================="
    echo "Running Logistic Regression for: $T"
    echo "========================================="
    
    $PLINK \
        --bfile $BFILE \
        --pheno $PHENO_FILE \
        --covar ${WORK_DIR}temp_covar_prs_${T}.txt \
        --logistic hide-covar \
        --out ${WORK_DIR}assoc_${T}
        
done

echo "All 9 logistic regression models have been generated."

Running Logistic Regression for: 5e-8
PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/assoc_5e-8.log.
Options in effect:
  --bfile /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/HapMap_3_r3_qc_final
  --covar /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/temp_covar_prs_5e-8.txt
  --logistic hide-covar
  --out /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/assoc_5e-8
  --pheno /Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/pheno.tsv

18432 MB RAM detected; reserving 9216 MB for main workspace.
1073788 variants loaded from .bim file.
76 people (39 males, 37 females) loaded from .fam.
76 phenotype values present after --p

Generating Zip file with all deliverables

In [78]:
%%bash

# Navigate to your working directory so the zip file doesn't include the whole folder path
WORK_DIR="/Users/kmlanderos/Documents/Johns_Hopkins/Spring_2026/Annotate_a_Genome/data/excercise4/"
cd $WORK_DIR

# Create the zip file and add all required outputs
zip PRS_Submission.zip \
    discovery_gwas_sim_sumstats_cleaned.tsv \
    prs_weights_harmonized.tsv \
    prs_weights_harmonized_qcd.tsv \
    clumped.snps \
    prs_*.snps \
    prs_*.profile \
    assoc_*.assoc.logistic


updating: discovery_gwas_sim_sumstats_cleaned.tsv (deflated 62%)
updating: prs_weights_harmonized_qcd.tsv (deflated 61%)
updating: prs_0.05.snps (deflated 59%)
updating: prs_0.1.snps (deflated 59%)
updating: prs_0.5.snps (deflated 59%)
updating: prs_1.0.snps (deflated 59%)
updating: prs_1e-2.snps (deflated 58%)
updating: prs_1e-3.snps (deflated 57%)
updating: prs_1e-4.snps (deflated 56%)
updating: prs_1e-6.snps (deflated 54%)
updating: prs_5e-8.snps (deflated 52%)
updating: prs_0.05.profile (deflated 70%)
updating: prs_0.1.profile (deflated 70%)
updating: prs_0.5.profile (deflated 70%)
updating: prs_1.0.profile (deflated 69%)
updating: prs_1e-2.profile (deflated 71%)
updating: prs_1e-3.profile (deflated 74%)
updating: prs_1e-4.profile (deflated 76%)
updating: prs_1e-6.profile (deflated 76%)
updating: prs_5e-8.profile (deflated 77%)
updating: assoc_0.05.assoc.logistic (deflated 80%)
updating: assoc_0.1.assoc.logistic (deflated 80%)
updating: assoc_0.5.assoc.logistic (deflated 80%)
updat